In [ ]:
# %% [1. ตรวจสอบสภาพแวดล้อมฮาร์ดแวร์]
import torch

# ตรวจสอบและเลือกใช้งานหน่วยประมวลผล GPU บนชิป M4
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"รันโมเดลบนฮาร์ดแวร์: {device}")

# %% [2. นำเข้าไลบรารีและตั้งค่าพารามิเตอร์หลัก]
import numpy as np
from scipy.io import loadmat
import torch.nn as nn
import math
import random
import gc
import os
from datetime import datetime as dt
from torch.utils.data import DataLoader, TensorDataset
import evaluate as ev
import visualizer as vis

ai_model = 'transformer'
scenario = 'O1'
antennas = 64
epochs = 100
seq_length = 5 # ระยะเวลา Sequence ของ Transformer
save_path = './best_models/'
os.makedirs(save_path, exist_ok=True)

# กำหนดสเปกตรัมความถี่และช่วง SNR สำหรับวิ่งลูปย่อยอัตโนมัติ 
frequencies = [28, 60, 140]
snr_list = [0, 5, 10, 15, 20] 

# ฟังก์ชันล็อคระบบ Seed เพื่อความแม่นยำในการเปรียบเทียบผลทดลอง
def apply_global_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)
    print(f"Global environment locked with seed: {seed}")

# ==========================================
# สถาปัตยกรรมโครงข่ายประสาทเทียมแบบ Transformer
# ==========================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(1), :].unsqueeze(0)
        return x

class BeamPredictionTransformer(nn.Module):
    def __init__(self, input_size, d_model, num_heads, num_layers, n_beams, dropout_rate):
        super(BeamPredictionTransformer, self).__init__()
        self.embedding = nn.Linear(input_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=num_heads, 
            dim_feedforward=d_model*4, dropout=dropout_rate, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(d_model, n_beams)

    def forward(self, x):
        x = torch.relu(self.embedding(x)) 
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x) 
        x = x[:, -1, :] 
        out = self.fc(x) 
        return out

# ฟังก์ชันจัดเตรียมข้อมูล Sequence ให้สอดคล้องกับ SE Data สำหรับ Transformer
def create_sequences(data, labels, se_data, seq_length):
    num_samples = len(data) - seq_length
    X_seq = np.zeros((num_samples, seq_length, data.shape[1]), dtype=np.float32)
    y_seq = np.zeros((num_samples, labels.shape[1]), dtype=np.float32)
    se_seq = np.zeros((num_samples, se_data.shape[1]), dtype=np.float32)
    
    for i in range(num_samples):
        X_seq[i] = data[i : i + seq_length]
        y_seq[i] = labels[i + seq_length - 1]
        se_seq[i] = se_data[i + seq_length - 1] # จัดการ SE ให้อยู่ลำดับเดียวกัน
    return X_seq, y_seq, se_seq

# %% [3. เริ่มต้นระบบประมวลผลลูป ทั่วทั้งตารางกริดทดลอง]
for frequency in frequencies:
    for snr in snr_list:
        print(f"\n" + "="*70)
        print(f"--- Start configuration: {ai_model.upper()} | Frequency {frequency} GHz | SNR {snr} dB ---")
        print("="*70)
        
        # 1. โหลดข้อมูลตามรอบโฟลเดอร์ปัจจุบันของตารางทดลอง
        path = f'../DeepMIMO/DeepMIMO/DeepMIMO_dataset/SNR{snr}dB_{scenario}_{frequency}_Ant{antennas}/'
        
        try:
            d1 = loadmat(path+'channel1.mat')['a']
            d2 = loadmat(path+'channel2.mat')['b']
            d3 = loadmat(path+'channel3.mat')['c']
        except FileNotFoundError:
            print(f"[ERROR] ไม่พบไฟล์ที่ {path} ข้ามไปลูปถัดไป...")
            continue

        data = np.concatenate((d1, d2, d3), axis=2).transpose(2, 0, 1)

        # จัดมิติข้อมูลให้เป็นเส้นตรง (Flatten) สำหรับ Transformer (4096 features)
        X_combined = np.concatenate((data.real, data.imag), axis=2).astype(np.float32)
        X_flattened = np.reshape(X_combined, (data.shape[0], -1))

        # การทำ Normalization ข้อมูลสัญญาณ
        mean = np.mean(X_flattened, axis=0)
        std = np.std(X_flattened, axis=0)
        X_flattened = (X_flattened - mean) / (std + 1e-8)

        print("Data normalization complete.")
        
        # โหลดไฟล์ข้อมูลฉลาก และ ข้อมูล Spectral Efficiency
        y = loadmat(path+'DLCB_output.mat')['onehot_label'].astype(np.float32)
        se_raw = loadmat(path+'rate_ave.mat')['DL_output'].astype(np.float32)

        # แปลงเป็น Sequence
        X_seq, y_seq, se_seq = create_sequences(X_flattened, y, se_raw, seq_length)
        print(f"Sequence created! Input shape: {X_seq.shape}")

        # 2. กระบวนการแบ่งส่วนข้อมูล (70% train, 30% test)
        split_idx = int(len(X_seq) * 0.7)

        X_train, X_test = X_seq[:split_idx], X_seq[split_idx:]
        y_train, y_test = y_seq[:split_idx], y_seq[split_idx:]
        se_test = se_seq[split_idx:]

        print(f"Data Split Complete:")
        print(f" - Training samples: {len(X_train)}")
        print(f" - Testing samples:  {len(X_test)}")

        train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
        test_ds = TensorDataset(torch.tensor(X_test), torch.tensor(y_test))

        train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
        test_loader = DataLoader(test_ds, batch_size=128, shuffle=False)

        # 3. ล็อคระบบความสุ่มทุกลูปเพื่อให้เริ่มฟิตจากจุดเดียวกัน
        apply_global_seed(42)

        # สร้างอินสแตนซ์โมเดล Transformer 
        # (คงพารามิเตอร์ให้ใกล้เคียง 100k Params เพื่อเปรียบเทียบ)
        model = BeamPredictionTransformer(
            input_size=4096, 
            d_model=22, 
            num_heads=2, 
            num_layers=1, 
            n_beams=64, 
            dropout_rate=0.2
        ).to(device)

        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)
        criterion = nn.CrossEntropyLoss()

        params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f'Total Trainable Params: {params}')

        # 4. ขั้นตอนการเทรน
        best_loss = float('inf')
        train_losses = []
        val_losses = []

        for epoch in range(epochs):
            # --- Training ---
            epoch_loss = 0.0
            model.train()
            
            for inputs, labels in train_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, torch.max(labels, 1)[1])
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
            
            avg_train_loss = epoch_loss / len(train_loader)
            train_losses.append(avg_train_loss)
            
            # --- Validation ---
            val_loss = 0.0
            model.eval()
            with torch.no_grad():
                for inputs, labels in test_loader:
                    inputs, labels = inputs.to(device), labels.to(device)
                    outputs = model(inputs)
                    loss = criterion(outputs, torch.max(labels, 1)[1])
                    val_loss += loss.item()
            
            avg_val_loss = val_loss / len(test_loader)
            val_losses.append(avg_val_loss)
                
            scheduler.step(avg_val_loss)
                
            if (epoch + 1) % 10 == 0:
                print(f'Epoch [{epoch+1}/{epochs}], Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, LR: {optimizer.param_groups[0]["lr"]:.6f}')

            # เซฟโมเดลโดยอิงจากคะแนน Validation เป็นหลัก
            if avg_val_loss < best_loss:
                best_loss = avg_val_loss
                torch.save(model.state_dict(), save_path+f'best_{ai_model}_model_snr{snr}_{scenario}_{frequency}ghz_{antennas}ant.pth')

        print(f"--> Finished Training. Best Validation Loss: {best_loss:.4f}")

        # 5. ประเมินผลและเรนเดอร์กราฟประสิทธิภาพ
        model.load_state_dict(torch.load(save_path+f'best_{ai_model}_model_snr{snr}_{scenario}_{frequency}ghz_{antennas}ant.pth'))
        model.eval()

        eval_datetime = dt.now().strftime("%Y-%m-%d %H:%M:%S")
        ds_config = {'snr': snr, 'scenario': scenario, 'frequency': frequency, 'antennas': antennas}  

        mimo_results = ev.evaluate_performance(model, test_loader, device, criterion, ai_model, ds_config, eval_datetime)
        
        # วาดกราฟสถิติพื้นฐาน
        vis.plot_training_loss(train_losses, val_losses, mimo_results, ds_config, eval_datetime)
        vis.plot_confusion_matrix(mimo_results['all_actuals'], mimo_results['all_preds'], ai_model, ds_config, eval_datetime)
        vis.plot_beam_tracking(mimo_results['all_actuals'], mimo_results['all_preds'], ai_model, ds_config, eval_datetime)
        
        # 6. คัดแยกและพล็อตกราฟประสิทธิภาพช่องสัญญาณ SE
        preds_array = np.array(mimo_results['all_preds'])
        actuals_array = np.array(mimo_results['all_actuals'])
        user_indices = np.arange(len(preds_array))
        
        predicted_se = se_test[user_indices, preds_array]
        optimal_se = se_test[user_indices, actuals_array]

        plot_limit = 200 # แสดงผลกราฟแค่ 200 ลำดับผู้ใช้แรก
        vis.plot_se_tracking(
            user_indices=user_indices[:plot_limit],
            optimal_se=optimal_se[:plot_limit],
            predicted_se=predicted_se[:plot_limit],
            model_name=ai_model, 
            ds_config=ds_config, 
            eval_datetime=eval_datetime
        )

        vis.save_se_summary_to_csv(mimo_results, se_test, ai_model, ds_config, eval_datetime)
        
        # 7. เคลียร์พื้นที่แคชบนระบบและเคลียร์ VRAM เพื่อป้องกันแรมเต็ม
        del d1, d2, d3, data, X_combined, X_flattened, y, se_raw
        del X_seq, y_seq, se_seq, X_train, X_test, y_train, y_test, se_test
        del train_ds, test_ds, train_loader, test_loader, model, optimizer, scheduler, criterion
        gc.collect()
        if torch.backends.mps.is_available():
            torch.mps.empty_cache()

print("\n" + "="*50)
print(f"=== ALL CONFIGURATIONS FOR {ai_model.upper()} COMPLETE ===")
print("="*50)